# Loading an existing model for inference with MindSpore

Let's load the exported model from our last notebook experiment for inference with MindSpore. We assume you have gone through our last experiment on predicting California house prices with MindSpore which exports the following model files.

1. `california-housing-linear-simple.mindir`: our model in MindIR format
1. `california-housing-linear-simple.onnx`: our model in ONNX format

Moreover, the model was trained on a transformed dataset with the following statistical values for their mean and standard deviation across both features and labels. This is crucial for us to properly "unwind" the predictions made by our model to recover the actual values relevant for our problem domain.

In [1]:
X_train_mean = 174.5399
X_train_std = 631.1331
y_train_log1p_mean = 1.0418
y_train_log1p_std = 0.3506

Let's first import MindSpore, initialize it for our platform and confirm it is correctly installed.

In [2]:
import mindspore
import os

NOTEBOOK_USE_ASCEND = os.getenv('NOTEBOOK_USE_ASCEND', '0')
NOTEBOOK_USE_GPU = os.getenv('NOTEBOOK_USE_GPU', '0')
NOTEBOOK_USE_CPU = os.getenv('NOTEBOOK_USE_CPU', '0')
NOTEBOOK_CI_MODE = os.getenv('NOTEBOOK_CI_MODE', '0')

platform = 'Ascend'
if NOTEBOOK_USE_ASCEND == '1':
    platform = 'Ascend'
elif NOTEBOOK_USE_GPU == '1':
    platform = 'GPU'
elif NOTEBOOK_USE_CPU == '1' or NOTEBOOK_CI_MODE == '1':
    platform = 'CPU'
else:
    platform = 'Ascend'

mindspore.set_device(device_target=platform)
mindspore.run_check()

/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:146: SyntaxWarning: invalid escape sequence '\c'
  2. In forward, tiling would not split c1 and c0, find c1\c0 based on t2.
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:172: SyntaxWarning: invalid escape sequence '\c'
  1. Forward: tiling would not split c1\c0\h0, find c1\c0\h1\h0 based on t2
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangepiaipro-20t/lib/python3.12/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangep

MindSpore version:  2.8.0
The result of multiplication calculation is correct, MindSpore has been installed on platform [Ascend] successfully!


It's safe to ignore any warnings produced unless you see `[ERROR]` or `[CRITICAL]` in which case consult the Ascend forum. The following output confirms that MindSpore is correctly installed.

```text
MindSpore version: 2.8.0
The result of multiplication calculation is correct, MindSpore has been installed on platfor [Ascend] successfully!
```

## Loading the California housing dataset

Let's load the California housing dataset for prediction using scikit-learn. We'll use just the last sample for inference.

In [3]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
X_sample = housing.data[-1]
y_sample = housing.target[-1]
X_sample.shape, y_sample.shape

((8,), ())

It is given that:

1. The features were transformed with `StandardScaler` prior to model training
1. The labels were transformed with the 2-step process below prior to model training
    1. Compute $\log (1 + y_i)$ for each label $y_i$
    1. Apply `StandardScaler` to the result in (a)

Let's define the following:

1. A function to transform our sample features before passing it to our model for prediction
1. An inverse function to "unwind" the transformation on the predicted label returned from our model

In [4]:
import numpy as np

def features_transform(X):
    X_scaled = (X - X_train_mean) / X_train_std
    return X_scaled

def label_inverse_transform(y_log1p_scaled):
    y_log1p = y_log1p_scaled * y_train_log1p_std + y_train_log1p_mean
    y = np.expm1(y_log1p)
    return y

Use `features_transform` to scale our sample features as we'll need it later.

In [5]:
X_sample_scaled = features_transform(X_sample)
X_sample_scaled

array([-0.27276544, -0.25119884, -0.26822422, -0.27470851,  1.92108463,
       -0.27240358, -0.2141702 , -0.46864901])

## Loading our model from MindIR format for inference

Loading our model from MindIR format for inference is a 2-step process.

1. Use the [`mindspore.load`](https://www.mindspore.cn/docs/en/r2.8.0/api_python/mindspore/mindspore.load.html) function to load the graph containing the model weights
1. Use the graph in \(1\) to instantiate [`mindspore.nn.GraphCell`](https://www.mindspore.cn/docs/en/r2.8.0/api_python/nn/mindspore.nn.GraphCell.html) for inference

In [6]:
import mindspore.nn as nn

mindir_graph = mindspore.load('california-housing-linear-simple.mindir')
mindir_model = nn.GraphCell(graph=mindir_graph)
mindir_model

GraphCell()

Let's predict the housing price for our input sample. Recall that our model was trained on transformed data so we need to pass in `X_sample_scaled` as our argument, and apply `label_inverse_transform` to the returned prediction.

In [7]:
y_hat_sample_log1p_scaled = mindir_model(mindspore.Tensor(X_sample_scaled.astype(np.float16).reshape(-1, 8)))
y_hat_sample = label_inverse_transform(y_hat_sample_log1p_scaled)
y_hat_sample_usd = y_hat_sample.item() * 100_000
y_sample_usd = y_sample.item() * 100_000
print(f'Predicted house price (USD$): {y_hat_sample_usd:.2f}')
print(f'Actual house price (USD$): {y_sample_usd:>12.2f}')
print(f'Percentage error: {(y_hat_sample_usd - y_sample_usd) / y_sample_usd * 100:>21.2f}%')

/usr/local/Ascend/cann-8.5.0/python/site-packages/asc_op_compile_base/asc_op_compiler/ascendc_compile_gen_code.py:161: SyntaxWarning: invalid escape sequence '\w'
  match = re.search(f'{option}=(\w+)', ' '.join(compile_options))
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/c

Predicted house price (USD$): 213476.56
Actual house price (USD$):     89400.00
Percentage error:                138.79%


Not too accurate, but you get the idea ;-\)

## Loading our model from ONNX format for inference

MindSpore does not support loading our model directly from ONNX format. Instead, use the following 2-step process.

1. Convert the ONNX model to MindIR format with [MindSpore Lite](https://www.mindspore.cn/lite/). MindSpore Lite is available as a separate Python package [`mindspore-lite`](https://pypi.org/project/mindspore-lite/)
1. Follow the usual process to load the converted model and initialize the `GraphCell` for inference

For brevity, we will skip the implementation. Just know that ONNX is fully supported by MindSpore and it can be done ;-\)